# Cluster Feature Importance: Impact of Clustering Method on Physiographic Predictors

This notebook asks: **does the choice of clustering method (SKATER vs Ward vs K-Means) affect which physiographic basin attributes are identified as important predictors?**

Two analyses:
- **Analysis A:** Random Forest classifier — how well can physiographic attributes predict cluster membership? Which features discriminate between clusters?
- **Analysis B:** Within-cluster RF regressor — does physiographic prediction of AEP return periods improve when sites are grouped by SKATER regions vs. unconstrained clusters?

**Clustering methods compared:** SKATER (k=22 kNN, floor=30), Ward hierarchical, K-Means — all at k=8 clusters, AEP threshold features.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rplot

plt.style.use('ryan')

from pathlib import Path
from shapely.geometry import Point
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import adjusted_rand_score
import scipy.stats as stats
import shap
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
DATA_DIR   = Path("/home/ryan/data/flood_hazard")
META_DIR   = Path("/home/ryan/data/flood_hazard/metadata")
NHD_DIR    = Path("/home/ryan/data/flood_hazard/NHD/US SCS")
GAGES2_DIR = Path("/home/ryan/data/usgs/GAGES_2/basinchar_and_report_sept_2011/spreadsheets-in-csv-format")
CLUST_DIR  = Path(".")
OUT_DIR    = Path(".")

N_CLUSTERS        = 8
MAX_DISTURB_INDEX = 15
WINSOR            = 0.02
RF_N_TREES        = 500
RF_SEED           = 42
N_PERM_REPEATS    = 30
CV_FOLDS          = 5
MIN_CLUSTER_N     = 30   # minimum complete-case sites per cluster for regression
SAVEFIG           = True

PREDICTORS = ['log_da', 'log_slope', 'log_bkfw', 'julaug_tempC',
              'prope', 'VBA_RWA_R', 'stream_order']

AEP_COLS = ["action_aep", "flood_aep", "moderate_aep", "major_aep"]
RP_FEATS = ["action_rp", "flood_rp", "moderate_rp", "major_rp"]
RP_SCALED = [f + "_s" for f in RP_FEATS]

CONUS_EXTENT = [-125, -66, 24, 50]

def make_conus_ax(fig, pos=111, title=''):
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,     facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,    linewidth=0.4, zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.6, zorder=3)
    if title:
        ax.set_title(title)
    return ax

---
## Section 1: Load and Merge All Data

### 1.1 AEP features and cluster labels

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "ffa" / "flood_frequency.parquet")
meta = pd.read_parquet(META_DIR / "site_info.parquet")[["site_no", "latitude", "longitude"]]

gages2 = pd.read_csv(GAGES2_DIR / "conterm_bas_classif.txt", encoding="latin1")
gages2["site_no"] = gages2["STAID"].astype(str).str.zfill(8)

df_aep = (
    ffa[ffa.record_ok & ~ffa.degenerate_fit]
    [["site_no"] + AEP_COLS]
    .dropna(subset=AEP_COLS)
    .merge(meta, on="site_no")
    .merge(gages2[["site_no", "HYDRO_DISTURB_INDX"]], on="site_no", how="left")
)
df_aep = df_aep[
    df_aep.HYDRO_DISTURB_INDX.notna() & (df_aep.HYDRO_DISTURB_INDX <= MAX_DISTURB_INDEX)
].reset_index(drop=True)

for aep_col, rp_col in zip(AEP_COLS, RP_FEATS):
    df_aep[rp_col] = np.log10(1.0 / df_aep[aep_col].clip(lower=1e-6))

df_clip = df_aep.copy()
for col in RP_FEATS:
    lo, hi = df_aep[col].quantile([WINSOR, 1 - WINSOR])
    df_clip[col] = df_aep[col].clip(lo, hi)

scaler = RobustScaler()
X_aep = scaler.fit_transform(df_clip[RP_FEATS])
for i, col in enumerate(RP_SCALED):
    df_aep[col] = X_aep[:, i]

print(f"AEP sites: {len(df_aep):,}")

In [ ]:
# SKATER — load from saved CSV
skater_labels = pd.read_csv(CLUST_DIR / "site_regions_skater_aep_8.csv")[["site_no", "cluster"]]
skater_labels["site_no"] = skater_labels["site_no"].astype(str).str.zfill(8)
skater_labels = skater_labels.rename(columns={"cluster": "cl_skater"})

# Ward unconstrained — re-run on the full df_aep
hc = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
df_aep["cl_ward"] = hc.fit_predict(X_aep) + 1

# K-Means — re-run
km = KMeans(n_clusters=N_CLUSTERS, random_state=RF_SEED, n_init=20)
df_aep["cl_kmeans"] = km.fit_predict(X_aep) + 1

# Merge SKATER labels
df_aep = df_aep.merge(skater_labels, on="site_no", how="inner")
X_aep = df_aep[RP_SCALED].values

print(f"Sites after SKATER inner join: {len(df_aep):,}")
for method in ["cl_skater", "cl_ward", "cl_kmeans"]:
    print(f"  {method}: {df_aep[method].value_counts().sort_index().to_dict()}")

### 1.2 Load physiographic attributes

In [ ]:
si = pd.read_parquet(META_DIR / "site_info.parquet",
                     columns=['site_no', 'drainage_area_sqmi', 'elevation_ft'])
cg = pd.read_parquet(META_DIR / "channel_geometry.parquet",
                     columns=['site_no', 'bankfull_width_ft', 'nhd_slope_ft_ft'])
gm = pd.read_parquet(META_DIR / "gage_map.parquet", columns=['site_no', 'reach_id'])

df_attrs = (si
            .merge(cg, on='site_no', how='left')
            .merge(gm, on='site_no', how='left'))
df_attrs['COMID'] = pd.to_numeric(df_attrs['reach_id'], errors='coerce').astype('Int64')

def load_nhd_table(files_spec, keep_cols):
    parts = []
    for path, renames in files_spec:
        d = pd.read_csv(path).rename(columns=renames)
        parts.append(d[[c for c in keep_cols if c in d.columns]])
    combined = pd.concat(parts, ignore_index=True)
    combined['COMID'] = pd.to_numeric(combined['COMID'], errors='coerce').astype('Int64')
    return combined.drop_duplicates('COMID')

SG = NHD_DIR / "Size_Gradient"
VC = NHD_DIR / "Valley_Confinement"
HY = NHD_DIR / "Hydrology"
TM = NHD_DIR / "Temperature"

nhd_sg = load_nhd_table([
    (SG / "East_SizeGradient.csv", {'StreamOrder': 'stream_order', 'StreamOrde': 'stream_order'}),
    (SG / "West_SizeGradient.csv", {'StreamOrder': 'stream_order', 'StreamOrde': 'stream_order'}),
    (SG / "UM_SizeGradient.csv",   {'StreamOrder': 'stream_order', 'StreamOrde': 'stream_order'}),
    (SG / "LM_SizeGradient.csv",   {'StreamOrder': 'stream_order', 'StreamOrde': 'stream_order'}),
], keep_cols=['COMID', 'stream_order'])
nhd_sg['stream_order'] = pd.to_numeric(nhd_sg['stream_order'], errors='coerce')

nhd_vc = load_nhd_table([
    (VC / "East_VC.csv", {'RWA': 'SWA'}),
    (VC / "West_VC.csv", {}),
    (VC / "UM_VC.csv",   {}),
    (VC / "LM_VC.csv",   {}),
], keep_cols=['COMID', 'VBA_RWA_R'])
nhd_vc = nhd_vc[nhd_vc['VBA_RWA_R'].notna()].copy()
nhd_vc['VBA_RWA_R'] = pd.to_numeric(nhd_vc['VBA_RWA_R'], errors='coerce')

nhd_hy = load_nhd_table([
    (HY / "East_hydro_classes.csv", {'prope1': 'prope'}),
    (HY / "West_hydro_classes.csv", {'prope1': 'prope'}),
    (HY / "UM_hydro_classes.csv",   {'prope1': 'prope'}),
    (HY / "LM_hydro_classes.csv",   {'prope1': 'prope'}),
], keep_cols=['COMID', 'prope'])

nhd_tm = load_nhd_table([
    (TM / "East_temp.csv", {'JulAug_tempC': 'julaug_tempC', 'JulyAug_tempC': 'julaug_tempC'}),
    (TM / "West_temp.csv", {'JulAug_tempC': 'julaug_tempC', 'JulyAug_tempC': 'julaug_tempC'}),
    (TM / "UM_temp.csv",   {'JulAug_tempC': 'julaug_tempC', 'JulyAug_tempC': 'julaug_tempC'}),
    (TM / "LM_temp.csv",   {'JulAug_tempC': 'julaug_tempC', 'JulyAug_tempC': 'julaug_tempC'}),
], keep_cols=['COMID', 'julaug_tempC'])
nhd_tm['julaug_tempC'] = pd.to_numeric(nhd_tm['julaug_tempC'], errors='coerce')

for tbl in [nhd_sg, nhd_vc, nhd_hy, nhd_tm]:
    df_attrs = df_attrs.merge(tbl, on='COMID', how='left')

df_attrs['log_da']    = np.log10(df_attrs['drainage_area_sqmi'].clip(lower=1e-3))
df_attrs['log_slope'] = np.log10(df_attrs['nhd_slope_ft_ft'].clip(lower=1e-6))
df_attrs['log_bkfw']  = np.log10(df_attrs['bankfull_width_ft'].clip(lower=1))

print("Predictor coverage:")
for p in PREDICTORS:
    n = df_attrs[p].notna().sum()
    print(f"  {p:15s}: {n:,} / {len(df_attrs):,}  ({100*n/len(df_attrs):.1f}%)")

### 1.3 Complete-cases merge

In [ ]:
df_complete = (
    df_attrs[['site_no'] + PREDICTORS]
    .dropna(subset=PREDICTORS)
    .merge(df_aep[['site_no', 'cl_skater', 'cl_ward', 'cl_kmeans'] + RP_FEATS],
           on='site_no', how='inner')
)

n_complete = len(df_complete)
print(f"Complete-case sites: {n_complete:,}")
for method in ['cl_skater', 'cl_ward', 'cl_kmeans']:
    counts = df_complete[method].value_counts().sort_index()
    print(f"  {method}: min={counts.min()}  max={counts.max()}  n_clusters={len(counts)}")

X_pred = df_complete[PREDICTORS].values

---
# Section 2: Analysis A — Cluster Predictability (RF Classifier)

Can physiographic basin attributes predict which cluster a site belongs to? If yes, which features discriminate most? Do the important features differ between clustering methods?

In [ ]:
clf_results = {}

for method, cl_col in [('SKATER', 'cl_skater'), ('Ward', 'cl_ward'), ('K-Means', 'cl_kmeans')]:
    y = df_complete[cl_col].values

    rf = RandomForestClassifier(
        n_estimators=RF_N_TREES, max_features='sqrt',
        min_samples_leaf=5, random_state=RF_SEED, n_jobs=-1
    )
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RF_SEED)
    cv_ba  = cross_val_score(rf, X_pred, y, cv=cv, scoring='balanced_accuracy')
    cv_acc = cross_val_score(rf, X_pred, y, cv=cv, scoring='accuracy')

    rf.fit(X_pred, y)

    perm = permutation_importance(
        rf, X_pred, y,
        n_repeats=N_PERM_REPEATS, random_state=RF_SEED,
        scoring='balanced_accuracy', n_jobs=-1
    )

    explainer = shap.TreeExplainer(rf)
    shap_vals = explainer.shap_values(X_pred)  # list of arrays (one per class)
    mean_abs_shap = np.abs(np.array(shap_vals)).mean(axis=(0, 1))

    clf_results[method] = {
        'cv_balanced_accuracy': cv_ba,
        'cv_accuracy': cv_acc,
        'model': rf,
        'perm_mean': perm.importances_mean,
        'perm_std':  perm.importances_std,
        'shap':      mean_abs_shap,
    }

    print(f"\n{method}:")
    print(f"  CV balanced_accuracy = {cv_ba.mean():.3f} ± {cv_ba.std():.3f}")
    print(f"  CV accuracy          = {cv_acc.mean():.3f} ± {cv_acc.std():.3f}")
    print(f"  Naive baseline (1/k) = {1/N_CLUSTERS:.3f}")

In [ ]:
# Save scores
score_rows = []
for method, res in clf_results.items():
    score_rows.append({
        'method': method,
        'cv_balanced_accuracy_mean': round(res['cv_balanced_accuracy'].mean(), 4),
        'cv_balanced_accuracy_std':  round(res['cv_balanced_accuracy'].std(),  4),
        'cv_accuracy_mean':          round(res['cv_accuracy'].mean(), 4),
        'naive_baseline':            round(1 / N_CLUSTERS, 4),
    })

scores_df = pd.DataFrame(score_rows)
scores_df.to_csv(OUT_DIR / 'cluster_predictability_scores.csv', index=False)
print(scores_df.to_string(index=False))

In [ ]:
# Sort features by SKATER importance for consistent ordering
skater_order = np.argsort(clf_results['SKATER']['perm_mean'])[::-1]
feat_sorted = [PREDICTORS[i] for i in skater_order]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
method_colors = [rplot.OKABE_ITO[1], rplot.OKABE_ITO[0], rplot.OKABE_ITO[2]]

for ax, (method, color) in zip(axes, zip(['SKATER', 'Ward', 'K-Means'], method_colors)):
    res = clf_results[method]
    means = [res['perm_mean'][i] for i in skater_order]
    stds  = [res['perm_std'][i]  for i in skater_order]
    ba    = res['cv_balanced_accuracy'].mean()

    ax.barh(feat_sorted, means, xerr=stds, color=color, alpha=0.8, capsize=3)
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_title(f"{method}\nbal. accuracy = {ba:.3f}", fontsize=9)
    ax.set_xlabel('Mean decrease in balanced_accuracy')

rplot.panel_labels(list(axes))
plt.suptitle(f'Permutation feature importance — physiographic attributes vs cluster membership  (n={n_complete:,})',
             fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'feature_importance_classifier.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP vs permutation importance comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, (method, color) in zip(axes, zip(['SKATER', 'Ward', 'K-Means'], method_colors)):
    res = clf_results[method]
    shap_sorted = [res['shap'][i] for i in skater_order]

    ax.barh(feat_sorted, shap_sorted, color=color, alpha=0.8)
    ax.set_title(f"{method} — mean |SHAP|", fontsize=9)
    ax.set_xlabel('Mean |SHAP value|')

rplot.panel_labels(list(axes))
plt.suptitle('SHAP feature importance — mean absolute SHAP value (averaged across classes)',
             fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'feature_importance_shap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Save importance table
imp_rows = []
for i, feat in enumerate(PREDICTORS):
    row = {'feature': feat}
    for method in ['SKATER', 'Ward', 'K-Means']:
        res = clf_results[method]
        row[f'{method}_perm_mean'] = round(res['perm_mean'][i], 5)
        row[f'{method}_perm_std']  = round(res['perm_std'][i], 5)
        row[f'{method}_shap']      = round(res['shap'][i], 5)
    imp_rows.append(row)

imp_df = pd.DataFrame(imp_rows)
imp_df.to_csv(OUT_DIR / 'feature_importance_perm.csv', index=False)
print(imp_df.to_string(index=False))

---
# Section 3: Analysis B — Within-Cluster Regression

For each clustering method, train an RF regressor within each cluster to predict AEP return periods from physiographic attributes. Compare mean CV-R² across methods and against a global (no-clustering) baseline.

In [ ]:
reg_results = []

methods_reg = [('SKATER', 'cl_skater'), ('Ward', 'cl_ward'), ('K-Means', 'cl_kmeans')]

for target_col in RP_FEATS:
    y_global = df_complete[target_col].values

    # Global baseline (no clustering)
    rf_global = RandomForestRegressor(
        n_estimators=RF_N_TREES, min_samples_leaf=5, random_state=RF_SEED, n_jobs=-1
    )
    kf_global = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RF_SEED)
    r2_global = cross_val_score(rf_global, X_pred, y_global, cv=kf_global, scoring='r2')
    reg_results.append({
        'method': 'Global', 'cluster': 'all', 'target': target_col,
        'n': n_complete, 'cv_r2_mean': r2_global.mean(), 'cv_r2_std': r2_global.std()
    })

    for method, cl_col in methods_reg:
        for cluster_id in sorted(df_complete[cl_col].unique()):
            mask = df_complete[cl_col] == cluster_id
            n = int(mask.sum())

            if n < MIN_CLUSTER_N:
                reg_results.append({
                    'method': method, 'cluster': cluster_id, 'target': target_col,
                    'n': n, 'cv_r2_mean': np.nan, 'cv_r2_std': np.nan
                })
                continue

            X_sub = X_pred[mask]
            y_sub = df_complete.loc[mask, target_col].values

            n_folds = min(CV_FOLDS, n // 10)
            if n_folds < 2:
                reg_results.append({
                    'method': method, 'cluster': cluster_id, 'target': target_col,
                    'n': n, 'cv_r2_mean': np.nan, 'cv_r2_std': np.nan
                })
                continue

            rf_sub = RandomForestRegressor(
                n_estimators=RF_N_TREES, min_samples_leaf=5,
                random_state=RF_SEED, n_jobs=-1
            )
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=RF_SEED)
            cv_r2 = cross_val_score(rf_sub, X_sub, y_sub, cv=kf, scoring='r2')

            reg_results.append({
                'method': method, 'cluster': cluster_id, 'target': target_col,
                'n': n, 'cv_r2_mean': cv_r2.mean(), 'cv_r2_std': cv_r2.std()
            })

reg_df = pd.DataFrame(reg_results)
reg_df.to_csv(OUT_DIR / 'within_cluster_r2.csv', index=False)
print(f"Saved within_cluster_r2.csv  ({len(reg_df)} rows)")

In [ ]:
# Weighted mean R² per method per target (weight by cluster n)
summary_rows = []
for target in RP_FEATS:
    for method in ['Global', 'SKATER', 'Ward', 'K-Means']:
        sub = reg_df[(reg_df['method'] == method) & (reg_df['target'] == target)].dropna(subset=['cv_r2_mean'])
        if len(sub) == 0:
            continue
        weights = sub['n'].values.astype(float)
        weighted_r2 = np.average(sub['cv_r2_mean'].values, weights=weights)
        summary_rows.append({
            'target': target, 'method': method,
            'n_clusters_valid': len(sub),
            'weighted_mean_r2': round(weighted_r2, 4)
        })

reg_summary = pd.DataFrame(summary_rows)
print(reg_summary.pivot(index='method', columns='target', values='weighted_mean_r2').to_string())

In [ ]:
methods_plot = ['Global', 'SKATER', 'Ward', 'K-Means']
colors_plot  = ['#888888'] + [rplot.OKABE_ITO[1], rplot.OKABE_ITO[0], rplot.OKABE_ITO[2]]
feat_labels  = ['Action', 'Flood', 'Moderate', 'Major']

x = np.arange(len(RP_FEATS))
width = 0.2
offsets = np.linspace(-(len(methods_plot)-1)/2 * width, (len(methods_plot)-1)/2 * width,
                      len(methods_plot))

fig, ax = plt.subplots(figsize=(11, 5))

for method, color, offset in zip(methods_plot, colors_plot, offsets):
    r2_vals = []
    for target in RP_FEATS:
        row = reg_summary[(reg_summary['method'] == method) & (reg_summary['target'] == target)]
        r2_vals.append(row['weighted_mean_r2'].values[0] if len(row) else np.nan)
    ax.bar(x + offset, r2_vals, width=width, label=method, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(feat_labels)
ax.set_xlabel('AEP threshold')
ax.set_ylabel('Weighted mean CV R²')
ax.set_title(f'Within-cluster RF regression: physiographic attributes → log₁₀(return period)\n(n={n_complete:,} complete-case sites, weights by cluster n)')
ax.legend(title='Method')
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'within_cluster_r2_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

---
# Section 4: Feature Importance Concordance

Kendall's tau between permutation importance rankings across clustering methods.
- tau > 0.8: physiographic drivers are robust to the clustering choice
- tau diverges for SKATER: the spatial constraint elicits different feature signals

In [ ]:
method_names = ['SKATER', 'Ward', 'K-Means']
rankings = {m: stats.rankdata(-clf_results[m]['perm_mean']) for m in method_names}

tau_matrix = np.eye(len(method_names))
for i, m1 in enumerate(method_names):
    for j, m2 in enumerate(method_names):
        if i != j:
            tau, p = stats.kendalltau(rankings[m1], rankings[m2])
            tau_matrix[i, j] = tau
            print(f"{m1} vs {m2}: τ={tau:.3f}  p={p:.4f}")

tau_df = pd.DataFrame(tau_matrix, index=method_names, columns=method_names)
print("\nKendall's τ matrix (permutation importance rankings):")
print(tau_df.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Tau heatmap
ax = axes[0]
im = ax.imshow(tau_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(method_names)))
ax.set_yticks(range(len(method_names)))
ax.set_xticklabels(method_names)
ax.set_yticklabels(method_names)
ax.set_title("Kendall's τ — feature importance rankings")
for i in range(len(method_names)):
    for j in range(len(method_names)):
        ax.text(j, i, f"{tau_matrix[i,j]:.2f}", ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

# Rank comparison scatter: SKATER vs Ward
ax = axes[1]
r_skater = rankings['SKATER']
r_ward   = rankings['Ward']
ax.scatter(r_skater, r_ward, s=60, color=rplot.OKABE_ITO[1], zorder=3)
for i, feat in enumerate(PREDICTORS):
    ax.annotate(feat, (r_skater[i], r_ward[i]), fontsize=7, xytext=(3, 3),
                textcoords='offset points')
ax.plot([1, len(PREDICTORS)], [1, len(PREDICTORS)], 'gray', linewidth=0.8, linestyle='--')
ax.set_xlabel('SKATER importance rank')
ax.set_ylabel('Ward importance rank')
ax.set_title('Importance rank: SKATER vs Ward')
ax.invert_xaxis()
ax.invert_yaxis()

rplot.panel_labels(list(axes))
plt.suptitle('Feature importance concordance across clustering methods', fontsize=10)
plt.tight_layout()
if SAVEFIG:
    plt.savefig(OUT_DIR / 'feature_importance_concordance.png', dpi=200, bbox_inches='tight')
plt.show()

---
# Interpretation

**Analysis A — Cluster predictability:**
- If SKATER's balanced accuracy is substantially *lower* than K-Means/Ward: physiographic attributes have weaker signal for spatially constrained regions, suggesting the spatial constraint is doing work that the attributes cannot replicate. This supports using SKATER — the geographic coherence it enforces is not redundant with the physiographic signal.
- If SKATER's balanced accuracy is *similar* to or higher than unconstrained methods: physiographic attributes already capture the geographic structure, and the spatial constraint is largely redundant.

**Analysis B — Within-cluster regression:**
- If SKATER clusters yield higher within-cluster R² than Ward/K-Means for the same target: the spatial contiguity constraint reduces within-cluster variance in a way that the physiographic predictors can explain, improving local predictability.
- If all methods perform similarly relative to the global baseline: the choice of clustering method matters less for the within-cluster prediction task.

**Kendall's τ:**
- High τ across all method pairs: the same physiographic features drive cluster membership regardless of the method — a robust finding.
- Low τ for SKATER vs unconstrained: the spatial constraint changes which features are important, suggesting SKATER clusters have a different physiographic character than the feature-space optimal clusters.